### Description

This notebook performs EDA on the downloaded data

In [12]:
import polars as pl
import plotly.graph_objects as go
import plotly.subplots as sp
import matplotlib.pyplot as plt
import ipywidgets as widgets




#### Weather data

In [13]:
path = "C:\\Users\\Arnold\\OneDrive\\Desktop\\CAPSTONE PROJECT\\farming_risk_regions\\data\\interim\\weather_data\\all_locations_weather_data.csv"

In [14]:
data = pl.read_csv(path)
data.head()

location,date,lat_center,lon_center,lat_min,lat_max,lon_min,lon_max,ALLSKY_SFC_SW_DWN,PRECTOTCORR,RH2M,T2M_MAX,T2M_MIN,WS10M
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Story_County_IA""","""2014-01-01""",42.025,-93.5,41.85,42.2,-93.81,-93.19,3.42,1.2,77.96,-13.07,-17.98,6.41
"""Story_County_IA""","""2014-01-02""",42.025,-93.5,41.85,42.2,-93.81,-93.19,8.17,0.0,77.08,-11.63,-18.55,4.37
"""Story_County_IA""","""2014-01-03""",42.025,-93.5,41.85,42.2,-93.81,-93.19,4.86,0.0,84.33,-2.26,-18.78,8.83
"""Story_County_IA""","""2014-01-04""",42.025,-93.5,41.85,42.2,-93.81,-93.19,3.2,0.01,72.45,-1.7,-12.34,8.15
"""Story_County_IA""","""2014-01-05""",42.025,-93.5,41.85,42.2,-93.81,-93.19,6.04,0.0,56.36,-13.14,-23.8,10.53


In [15]:
data.tail()

location,date,lat_center,lon_center,lat_min,lat_max,lon_min,lon_max,ALLSKY_SFC_SW_DWN,PRECTOTCORR,RH2M,T2M_MAX,T2M_MIN,WS10M
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Champaign_County_IL""","""2024-12-27""",40.15,-88.175,39.95,40.35,-88.4,-87.95,2.05,2.72,94.34,11.94,5.91,6.56
"""Champaign_County_IL""","""2024-12-28""",40.15,-88.175,39.95,40.35,-88.4,-87.95,6.75,3.9,92.73,13.67,5.83,3.26
"""Champaign_County_IL""","""2024-12-29""",40.15,-88.175,39.95,40.35,-88.4,-87.95,2.78,20.8,91.77,9.89,3.37,7.98
"""Champaign_County_IL""","""2024-12-30""",40.15,-88.175,39.95,40.35,-88.4,-87.95,6.18,4.7,90.29,6.51,1.04,3.91
"""Champaign_County_IL""","""2024-12-31""",40.15,-88.175,39.95,40.35,-88.4,-87.95,2.48,8.66,89.41,5.5,-1.09,8.33


In [16]:
## Removing duplicate rows if any
data = data.unique()
data.shape

(44198, 14)

In [17]:
data.describe()

statistic,location,date,lat_center,lon_center,lat_min,lat_max,lon_min,lon_max,ALLSKY_SFC_SW_DWN,PRECTOTCORR,RH2M,T2M_MAX,T2M_MIN,WS10M
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""44198""","""44198""",44198.0,44198.0,44198.0,44198.0,44198.0,44198.0,44198.0,44198.0,44198.0,44198.0,44198.0,44198.0
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,41.327727,-92.131818,41.105455,41.55,-92.44,-91.823636,14.719833,2.454975,74.375474,16.304562,5.176957,4.775488
"""std""",null,null,1.370154,3.739818,1.368966,1.372803,3.75981,3.720255,7.90042,5.876122,12.246216,12.012952,11.079028,1.913279
"""min""","""Champaign_County_IL""","""2014-01-01""",39.175,-99.675,39.0,39.35,-100.0,-99.35,0.76,0.0,28.96,-25.27,-38.65,0.67
"""25%""",null,null,40.53,-95.175,40.21,40.85,-95.5,-94.85,8.08,0.0,66.3,6.52,-3.1,3.37
"""50%""",null,null,40.925,-93.225,40.7,41.15,-93.5,-92.95,14.01,0.08,75.42,18.06,5.24,4.5
"""75%""",null,null,42.025,-88.6,41.85,42.2,-88.9,-88.3,21.3,1.76,83.65,26.79,15.1,5.86
"""max""","""Webster_County_IA""","""2024-12-31""",44.725,-87.7,44.5,44.95,-87.95,-87.45,32.47,90.35,99.99,42.71,26.17,15.68


In [18]:
## Changing the date from string to date type
data = data.with_columns(pl.col("date").str.strptime(pl.Date, format="%Y-%m-%d"))

In [19]:
locations = data['location'].unique().to_list()

dropdown = widgets.Dropdown(options=locations, value=locations[0], description='Location:')

def plot_weather(location):
    df = data.filter(pl.col('location') == location)
    fig = sp.make_subplots(rows=3,
                           cols=2, 
                           subplot_titles=("Infrared Radiation Distribution", 
                                            "Precipitation Distribution", 
                                            "Humidity Distribution",
                                            "Max Temperature Distribution", 
                                            "Min Temperature Distribution", 
                                            "Wind Speed Distribution"))
    fig.add_trace(go.Histogram(x=df['ALLSKY_SFC_SW_DWN'], nbinsx=50, name='Infrared Radiation'), row=1, col=1)
    fig.add_trace(go.Histogram(x=df['PRECTOTCORR'], nbinsx=50, name='Precipitation'), row=1, col=2)
    fig.add_trace(go.Histogram(x=df['RH2M'], nbinsx=50, name='Humidity'), row=3, col=2)
    fig.add_trace(go.Histogram(x=df['T2M_MAX'], nbinsx=50, name='Max Temperature'), row=2, col=1)
    fig.add_trace(go.Histogram(x=df['T2M_MIN'], nbinsx=50, name='Min Temperature'), row=2, col=2)
    fig.add_trace(go.Histogram(x=df['WS10M'], nbinsx=50, name='Wind Speed'), row=3, col=1)
    fig.update_layout(height=900, width=900, title_text=f"{location} Weather Parameter Distributions")
    fig.show()

widgets.interactive(plot_weather, location=dropdown)

interactive(children=(Dropdown(description='Location:', options=('LaSalle_County_IL', 'Pottawattamie_County_IA…

In [ ]:
# Create a dropdown for selecting a location
location_dropdown = widgets.Dropdown(
    options=locations,
    value=locations[0],
    description='Location:'
)

# Function to plot time series for the selected location
def plot_time_series(location):
    # Filter data for the selected location
    filtered_data = data.filter(pl.col('location') == location)
    
    # Group by date and aggregate (in case there are duplicates)
    # Also extract year-month for better visualization
    filtered_data = filtered_data.with_columns([
        pl.col('date').dt.year().alias('year'),
        pl.col('date').dt.month().alias('month')
    ])
    
    # Group by year and month to get monthly averages
    monthly_data = filtered_data.group_by(['year', 'month']).agg([
        pl.col('ALLSKY_SFC_SW_DWN').mean().alias('ALLSKY_SFC_SW_DWN'),
        pl.col('PRECTOTCORR').sum().alias('PRECTOTCORR'),  # Sum for precipitation
        pl.col('RH2M').mean().alias('RH2M'),
        pl.col('T2M_MAX').mean().alias('T2M_MAX'),
        pl.col('T2M_MIN').mean().alias('T2M_MIN'),
        pl.col('WS10M').mean().alias('WS10M')
    ]).sort(['year', 'month'])
    
    # Create a date column for plotting
    monthly_data = monthly_data.with_columns(
        pl.date(pl.col('year'), pl.col('month'), 1).alias('plot_date')
    )
    
    fig = sp.make_subplots(rows=3,
                           cols=2, 
                           subplot_titles=("Infrared Radiation (Monthly Avg)", 
                                           "Precipitation (Monthly Total)", 
                                           "Max Temperature (Monthly Avg)",
                                           "Min Temperature (Monthly Avg)", 
                                           "Humidity (Monthly Avg)",
                                           "Wind Speed (Monthly Avg)"))

    fig.add_trace(go.Scatter(x=monthly_data['plot_date'], 
                            y=monthly_data['ALLSKY_SFC_SW_DWN'], 
                            name='Infrared Radiation',
                            mode='lines'), row=1, col=1)
    
    fig.add_trace(go.Scatter(x=monthly_data['plot_date'], 
                            y=monthly_data['PRECTOTCORR'], 
                            name='Precipitation',
                            mode='lines'), row=1, col=2)
    
    fig.add_trace(go.Scatter(x=monthly_data['plot_date'], 
                            y=monthly_data['T2M_MAX'], 
                            name='Max Temperature',
                            mode='lines'), row=2, col=1)
    
    fig.add_trace(go.Scatter(x=monthly_data['plot_date'], 
                            y=monthly_data['T2M_MIN'], 
                            name='Min Temperature',
                            mode='lines'), row=2, col=2)
    
    fig.add_trace(go.Scatter(x=monthly_data['plot_date'], 
                            y=monthly_data['RH2M'], 
                            name='Humidity',
                            mode='lines'), row=3, col=1)
    
    fig.add_trace(go.Scatter(x=monthly_data['plot_date'], 
                            y=monthly_data['WS10M'], 
                            name='Wind Speed',
                            mode='lines'), row=3, col=2)

    fig.update_layout(height=900, 
                     width=1200, 
                     title_text=f"{location} Weather Parameter Time Series (Monthly)",
                     showlegend=False)
    
    # Update x-axes labels
    fig.update_xaxes(title_text="Date")
    
    fig.show()

# Create and display the interactive widget
display(widgets.interactive(plot_time_series, location=location_dropdown))

interactive(children=(Dropdown(description='Location:', options=('LaSalle_County_IL', 'Pottawattamie_County_IA…